# Práctica 11 · Del piloto a producción

Esta práctica es distinta de las anteriores. No vas a construir nada nuevo: vas a decidir.

Lo que has armado hasta ahora es un piloto. Funciona, está medido y sirve para demostrar que la
idea es viable. Lo que no es todavía es un sistema en el que una empresa apoye su atención al
cliente, y la distancia entre las dos cosas no se cubre programando más. Se cubre planeando:
poniendo números a lo que hoy son intuiciones, decidiendo qué se compra y qué se construye,
calculando cuánto va a costar y acordando de antemano qué se va a considerar un fracaso.

Aquí no hace falta Ollama ni ningún modelo. Todo se calcula con aritmética, y esa es en sí misma
una observación sobre los proyectos de este tipo: la parte que decide si salen bien rara vez es
la parte técnica.

Al terminar vas a tener un documento con el plan, generado por el propio cuaderno, que puedes
usar como base de una propuesta real.

Las celdas se ejecutan en orden, una por una, con Shift + Enter.

## 1. Empezar por lo que ya sabes

Todo plan de producción arranca con un informe de lo que dejó el piloto. No es un trámite: es la
única fuente de datos que tienes que no sean supuestos, y es lo que te permite discutir con
números en vez de con opiniones.

Las preguntas que ese informe tiene que contestar son siempre las mismas. Qué componentes se
usaron. De dónde salieron los datos y qué costó prepararlos. Qué tan bien respondió, medido y no
opinado. Qué problemas aparecieron que nadie había previsto. Y qué faltó, con la razón por la que
faltó.

Empecemos por llenar el inventario de componentes de tu piloto.

In [1]:
# Una lista de tríos: pieza, qué se eligió y por qué. La tercera columna es la que
# importa en una reunión: una decisión sin su razón no se puede defender ni revisar.
COMPONENTES = [
    ("Modelo de lenguaje",  "gemma3:4b, local con Ollama",
     "Elegido para que corra en cualquier laptop del curso"),
    ("Modelo de embeddings", "embeddinggemma:300m",
     "Único de los tres probados que cruza español e inglés"),
    ("Base vectorial",       "LanceDB, embebida en disco",
     "Sin servidor que administrar; suficiente a esta escala"),
    ("Fragmentación",        "recursiva, 500 caracteres con 100 de traslape",
     "La que menos supuestos hace sobre el formato del documento"),
    ("Búsqueda",             "vectorial densa, k=3",
     "La híbrida se probó y no mejoró con este corpus"),
    ("Guardrails",           "instrucción de abstención y filtro determinista",
     "La defensa por instrucción sola resultó insuficiente"),
    ("Interfaz",             "Gradio, con citas y pulgar",
     "Suficiente para el piloto, no para producción"),
    ("Evaluación",           "gold set de 28 preguntas por tipo",
     "Construido a mano; es lo que sostiene todas las cifras"),
]

print("INVENTARIO DEL PILOTO\n")
print(f"{'componente':<22} {'elección':<44} {'por qué'}")
print("-" * 110)
for parte, eleccion, razon in COMPONENTES:
    print(f"{parte:<22} {eleccion:<44} {razon}")

INVENTARIO DEL PILOTO

componente             elección                                     por qué
--------------------------------------------------------------------------------------------------------------
Modelo de lenguaje     gemma3:4b, local con Ollama                  Elegido para que corra en cualquier laptop del curso
Modelo de embeddings   embeddinggemma:300m                          Único de los tres probados que cruza español e inglés
Base vectorial         LanceDB, embebida en disco                   Sin servidor que administrar; suficiente a esta escala
Fragmentación          recursiva, 500 caracteres con 100 de traslape La que menos supuestos hace sobre el formato del documento
Búsqueda               vectorial densa, k=3                         La híbrida se probó y no mejoró con este corpus
Guardrails             instrucción de abstención y filtro determinista La defensa por instrucción sola resultó insuficiente
Interfaz               Gradio, con citas y pulgar       

Ese "por qué" de la tercera columna es lo que casi nunca se escribe y lo que más falta hace
después. Dentro de seis meses, cuando alguien proponga cambiar el modelo de embeddings, la
pregunta va a ser por qué se eligió el actual. Si la respuesta está escrita y además está medida,
la conversación dura cinco minutos. Si no, se repite el trabajo.

## 2. Los números que dejó el piloto

Aquí van las mediciones de las prácticas anteriores, reunidas. Conviene tenerlas juntas porque es
la primera columna de la tabla de requisitos que vamos a armar enseguida.

In [2]:
# Un diccionario guarda pares de nombre y valor. Aquí están TODAS las cifras medidas
# en las prácticas anteriores, juntas en un solo lugar. Ninguna es inventada, y por
# eso el resto del cuaderno puede calcular sobre ellas en vez de suponer.
PILOTO = {
    # Volumen
    "documentos": 17,
    "fragmentos": 85,
    # Tiempo, en milisegundos
    "ms_vectorizar": 97,
    "ms_buscar": 5,
    "ms_generar": 1685,
    "fragmentos_por_segundo_ingesta": 81,
    # Calidad de la recuperación
    "precision_k3": 0.768,
    "cobertura_k3": 0.848,
    "mrr": 0.877,
    # Calidad de la respuesta
    "sin_nada_incorrecto": 0.464,
    "se_abstiene": 4 / 5,
    # Consumo
    "tokens_entrada": 607,
    "tokens_salida": 48,
}

# El tiempo total de una consulta es la suma de sus tres etapas.
total_ms = PILOTO["ms_vectorizar"] + PILOTO["ms_buscar"] + PILOTO["ms_generar"]

print("RESUMEN DE MEDICIONES DEL PILOTO\n")
print(f"  corpus                        {PILOTO['documentos']} documentos, "
      f"{PILOTO['fragmentos']} fragmentos")
print(f"  tiempo por consulta           {total_ms/1000:.1f} s")
print(f"     de eso, generar la respuesta   {PILOTO['ms_generar']/total_ms*100:.0f}%")
print(f"     buscar en el índice            {PILOTO['ms_buscar']/total_ms*100:.0f}%")
print(f"  precisión de la búsqueda      {PILOTO['precision_k3']:.0%}")
print(f"  cobertura de la búsqueda      {PILOTO['cobertura_k3']:.0%}")
print(f"  el documento correcto primero {PILOTO['mrr']:.0%}")
print(f"  respuestas sin nada incorrecto {PILOTO['sin_nada_incorrecto']:.0%}")
print(f"  se abstiene cuando debe       {PILOTO['se_abstiene']:.0%}")
print(f"  tokens por consulta           {PILOTO['tokens_entrada']} + {PILOTO['tokens_salida']}")

RESUMEN DE MEDICIONES DEL PILOTO

  corpus                        17 documentos, 85 fragmentos
  tiempo por consulta           1.8 s
     de eso, generar la respuesta   94%
     buscar en el índice            0%
  precisión de la búsqueda      77%
  cobertura de la búsqueda      85%
  el documento correcto primero 88%
  respuestas sin nada incorrecto 46%
  se abstiene cuando debe       80%
  tokens por consulta           607 + 48


Hay un dato de esa lista que decide casi todo lo que viene después: **el 94% del tiempo se va en
redactar la respuesta**. Buscar en el índice cuesta cinco milisegundos.

Eso significa que si mañana te piden que el sistema responda más rápido, cambiar la base
vectorial no va a servir de nada. La palanca está en el modelo que genera, o en no generar, que
es lo que hace la caché.

Es el tipo de conclusión que solo se tiene si se midió por etapas, y es la diferencia entre
invertir donde duele e invertir donde es más fácil de justificar.

## 3. La tabla de requisitos

Este es el corazón del plan. Se trata de escribir, para cada aspecto del sistema, dónde está hoy
y dónde tiene que estar para poder salir a producción.

Dos advertencias antes de llenarla.

La primera es que las metas tienen que ser números, no adjetivos. "Que responda rápido" no es una
meta; "que el 95% de las consultas se responda en menos de 3 segundos" sí lo es, porque se puede
verificar y porque obliga a discutir si ese número es el correcto.

La segunda es que hay filas donde la meta correcta es dejar las cosas como están. Subir una
métrica siempre cuesta, y si el piloto ya está en un nivel aceptable, ese esfuerzo rinde más en
otra fila. Vas a ver una de esas abajo.

In [3]:
# Cada fila compara dónde está hoy el piloto contra dónde tiene que llegar, con una
# prioridad. Escribir el estado actual con números y no con adjetivos es lo que
# convierte esta tabla en un plan y no en una lista de deseos.
REQUISITOS = [
    ("Tiempo de respuesta",
     f"{total_ms/1000:.1f} s en promedio",
     "95% por debajo de 3 s",
     "alta"),
    ("Disponibilidad",
     "sin medir; corre en una laptop",
     "99.5% en horario de atención",
     "alta"),
    ("Cobertura de la búsqueda",
     f"{PILOTO['cobertura_k3']:.0%}",
     "90%",
     "media"),
    ("Respuestas sin nada incorrecto",
     f"{PILOTO['sin_nada_incorrecto']:.0%}",
     "85%",
     "alta"),
    ("Abstenerse cuando no sabe",
     f"{PILOTO['se_abstiene']:.0%} sobre 5 casos",
     "100% sobre 40 casos",
     "crítica"),
    ("Detección de datos personales",
     "medida sobre 5 tickets",
     "revisada sobre 100 tickets reales",
     "crítica"),
    ("Control de acceso por rol",
     "probado con 3 roles y 6 documentos",
     "integrado al directorio de la empresa",
     "alta"),
    ("Fuentes de datos",
     "17 archivos locales",
     "centro de ayuda, catálogo y base de pedidos",
     "alta"),
    ("Actualización del corpus",
     "manual, se reconstruye todo",
     "incremental, diaria y automática",
     "media"),
    ("Idiomas",
     "español",
     "español, con inglés en catálogo",
     "baja"),
    ("Registro de consultas",
     "en memoria, se pierde al cerrar",
     "persistente, con retención de 12 meses",
     "alta"),
    ("Gold set de evaluación",
     "28 preguntas",
     "150 preguntas de consultas reales",
     "alta"),
]

print("DE DÓNDE A DÓNDE\n")
print(f"{'aspecto':<32} {'hoy en el piloto':<34} {'meta en producción':<40} {'prioridad'}")
print("-" * 120)
for aspecto, hoy, meta, prioridad in REQUISITOS:
    print(f"{aspecto:<32} {hoy:<34} {meta:<40} {prioridad}")

# Esta línea filtra la lista quedándose solo con las filas cuya cuarta columna (r[3])
# dice "crítica". Son las que bloquean la salida a producción.
criticas = [r for r in REQUISITOS if r[3] == "crítica"]
print(f"\n{len(criticas)} requisitos marcados como críticos. Son los que, si no se cumplen,")
print("impiden salir a producción aunque todo lo demás esté listo:")
for aspecto, hoy, meta, _ in criticas:
    print(f"   {aspecto}: de '{hoy}' a '{meta}'")

DE DÓNDE A DÓNDE

aspecto                          hoy en el piloto                   meta en producción                       prioridad
------------------------------------------------------------------------------------------------------------------------
Tiempo de respuesta              1.8 s en promedio                  95% por debajo de 3 s                    alta
Disponibilidad                   sin medir; corre en una laptop     99.5% en horario de atención             alta
Cobertura de la búsqueda         85%                                90%                                      media
Respuestas sin nada incorrecto   46%                                85%                                      alta
Abstenerse cuando no sabe        80% sobre 5 casos                  100% sobre 40 casos                      crítica
Detección de datos personales    medida sobre 5 tickets             revisada sobre 100 tickets reales        crítica
Control de acceso por rol        probado con 3 role

Fíjate en la fila de la cobertura de búsqueda. El piloto está en 85% y la meta propuesta es 90%,
un salto modesto, con prioridad media. Podría parecer poco ambicioso.

Es deliberado, y la razón está en las mediciones de la práctica anterior: la cobertura ya es alta
mientras que las respuestas completas y correctas están por debajo de la mitad. Es decir, la
información está llegando y el problema es lo que el modelo hace con ella. Poner el esfuerzo en
subir la cobertura de 85 a 95 sería trabajar en la parte que ya funciona.

Ese razonamiento es el que hay que poder defender frente a quien financia el proyecto, y es la
razón por la que la práctica de evaluación va antes que esta.

Las filas marcadas como críticas merecen atención aparte. Las dos primeras son de seguridad, y
tienen una característica que las distingue del resto: no se negocian a cambio de tiempo. Una
respuesta lenta molesta; una fuga de datos personales es un problema legal. Cuando haya que
recortar alcance por calendario, estas son las que no se tocan.

## 4. Cuánta carga tiene que aguantar

Los requisitos de arriba se vuelven decisiones técnicas en cuanto les pones volumen. Vamos a
convertir el tráfico esperado en las cifras que necesitas para dimensionar.

In [4]:
# Ahora se pasa del piloto a la operación real. Estos números son suposiciones del
# negocio, no mediciones: cámbielos por los de su caso y todo lo de abajo se recalcula.
OPERACION = {
    "consultas_por_dia": 800,
    "dias_al_mes": 30,
    "hora_pico_pct": 0.18,   # proporción de las consultas del día en la hora más cargada
    "documentos_del_corpus": 450,
    "fragmentos_por_documento": 5,
    "documentos_que_cambian_al_dia": 12,
}

# Lo que hay que dimensionar no es el promedio, sino el pico: el sistema tiene que
# aguantar la hora más cargada del día, no la media del mes.
consultas_mes = OPERACION["consultas_por_dia"] * OPERACION["dias_al_mes"]
pico_por_hora = OPERACION["consultas_por_dia"] * OPERACION["hora_pico_pct"]
pico_por_segundo = pico_por_hora / 3600
fragmentos = OPERACION["documentos_del_corpus"] * OPERACION["fragmentos_por_documento"]

print("DIMENSIONAMIENTO\n")
print(f"  consultas al mes                    {consultas_mes:,}")
print(f"  consultas en la hora pico           {pico_por_hora:.0f}")
print(f"  consultas por segundo en el pico    {pico_por_segundo:.2f}")
print(f"\n  fragmentos en el índice             {fragmentos:,}")
print(f"     (el piloto tenía {PILOTO['fragmentos']}, o sea "
      f"{fragmentos/PILOTO['fragmentos']:.0f} veces menos)")

# Cuántas consultas están en curso a la vez: llegadas por segundo por lo que tarda
# cada una. Si el número es menor que uno, ni siquiera se solapan.
segundos_por_consulta = total_ms / 1000
consultas_simultaneas = pico_por_segundo * segundos_por_consulta
print(f"\n  consultas atendiéndose a la vez en el pico: {consultas_simultaneas:.2f}")

# Con el ritmo medido en la práctica 6 se puede comparar reconstruir todo el índice
# contra actualizar solo lo que cambió. La diferencia decide la operación diaria.
reindexado = fragmentos / PILOTO["fragmentos_por_segundo_ingesta"]
incremental = (OPERACION["documentos_que_cambian_al_dia"]
               * OPERACION["fragmentos_por_documento"]
               / PILOTO["fragmentos_por_segundo_ingesta"])
print(f"\n  reconstruir el índice completo      {reindexado:.0f} s")
print(f"  actualizar solo lo que cambió       {incremental:.1f} s")
print(f"  la actualización incremental es {reindexado/incremental:.0f} veces más barata")

DIMENSIONAMIENTO

  consultas al mes                    24,000
  consultas en la hora pico           144
  consultas por segundo en el pico    0.04

  fragmentos en el índice             2,250
     (el piloto tenía 85, o sea 26 veces menos)

  consultas atendiéndose a la vez en el pico: 0.07

  reconstruir el índice completo      28 s
  actualizar solo lo que cambió       0.7 s
  la actualización incremental es 38 veces más barata


Estos números suelen sorprender, y casi siempre en la misma dirección: la carga es mucho menor de
lo que la gente imagina.

Ochocientas consultas al día, que para una tienda mediana es tráfico real, son menos de una
consulta cada dos segundos en la hora más cargada. Con una respuesta de dos segundos, eso
significa que rara vez habrá más de una consulta atendiéndose a la vez.

La consecuencia práctica es que a esta escala no hace falta ninguna arquitectura sofisticada. La
tentación de diseñar para millones de consultas es fuerte y es cara; el número de arriba es el
antídoto.

Lo del reindexado va en el mismo sentido pero con más filo. Reconstruir el índice entero toma
menos de un minuto, así que técnicamente podrías hacerlo cada noche y olvidarte del asunto. La
razón para hacerlo incremental no es el tiempo de cómputo, es lo que ya viste en la práctica de
ingesta: mientras se reconstruye, el índice está a medias, y una consulta que llegue en ese
momento recibe una respuesta incompleta sin que nada avise.

Conviene tener presente que esos veintiocho segundos salen de una proyección: son los fragmentos
del corpus previsto divididos entre el ritmo de vectorizado que mediste. Si el corpus real
resulta ser diez veces mayor, o si los documentos traen tablas e imágenes que hay que procesar
aparte, la cuenta cambia. La proyección sirve para decidir, no para prometer.

## 5. El presupuesto

Ahora el dinero, que es donde estos proyectos se aprueban o se archivan.

Hay dos cosas que separar y que suelen mezclarse. Una es el costo de operar el sistema, que es lo
que casi todo el mundo calcula. La otra es el costo del trabajo humano de mantenerlo, que casi
nadie calcula y que suele ser mayor.

In [5]:
# El costo tiene dos partes que casi nunca se presentan juntas: el cómputo, que es
# fácil de cotizar, y las horas de personas, que es donde de verdad se va el dinero.
COSTOS = {
    # Operación mensual, en dólares
    "api_por_consulta": 0.000120,      # medido en la práctica de costos
    "local_equipo_mes": 55.56,         # equipo amortizado a 36 meses
    "local_electricidad_mes": 0.50,
    # Trabajo humano, en horas al mes
    "horas_curar_corpus": 16,
    "horas_revisar_evaluacion": 8,
    "horas_atender_incidencias": 6,
    "costo_hora": 25.0,
}

api_mes = consultas_mes * COSTOS["api_por_consulta"]
local_mes = COSTOS["local_equipo_mes"] + COSTOS["local_electricidad_mes"]
# Las tres tareas humanas que no desaparecen: mantener el corpus al día, revisar la
# evaluación y atender lo que falle. Un presupuesto que las omita se queda corto.
horas = (COSTOS["horas_curar_corpus"] + COSTOS["horas_revisar_evaluacion"]
         + COSTOS["horas_atender_incidencias"])
personas_mes = horas * COSTOS["costo_hora"]

print("COSTO MENSUAL\n")
print(f"{'concepto':<40} {'USD':>10}")
print("-" * 52)
print(f"{'cómputo con API de pago':<40} {api_mes:>10.2f}")
print(f"{'cómputo con modelo local':<40} {local_mes:>10.2f}")
print("-" * 52)
print(f"{'trabajo humano (' + str(horas) + ' h/mes)':<40} {personas_mes:>10.2f}")
print("-" * 52)
menor = min(api_mes, local_mes)
print(f"{'TOTAL con la opción de cómputo más barata':<40} {menor + personas_mes:>10.2f}")
print(f"\n  el cómputo es el {menor/(menor+personas_mes)*100:.1f}% del costo total")
print(f"  el trabajo humano es el {personas_mes/(menor+personas_mes)*100:.1f}%")

print("\n\n¿A partir de cuánto tráfico conviene el modelo local?\n")
print(f"{'consultas al mes':>18} {'API':>10} {'local':>10} {'conviene':>10}")
print("-" * 52)
for n in (5_000, 24_000, 100_000, 460_000, 500_000):
    a = n * COSTOS["api_por_consulta"]
    print(f"{n:>18,} {a:>10.2f} {local_mes:>10.2f} "
          f"{'API' if a < local_mes else 'local':>10}")
# El volumen a partir del cual el modelo local sale más barato. Compárelo con el
# volumen proyectado antes de presentar lo local como un ahorro.
equilibrio = local_mes / COSTOS["api_por_consulta"]
print(f"\n  se igualan alrededor de {equilibrio:,.0f} consultas al mes")
print(f"  tu operación proyectada es de {consultas_mes:,}, es decir "
      f"{equilibrio/consultas_mes:.0f} veces menos")

COSTO MENSUAL

concepto                                        USD
----------------------------------------------------
cómputo con API de pago                        2.88
cómputo con modelo local                      56.06
----------------------------------------------------
trabajo humano (30 h/mes)                    750.00
----------------------------------------------------
TOTAL con la opción de cómputo más barata     752.88

  el cómputo es el 0.4% del costo total
  el trabajo humano es el 99.6%


¿A partir de cuánto tráfico conviene el modelo local?

  consultas al mes        API      local   conviene
----------------------------------------------------
             5,000       0.60      56.06        API
            24,000       2.88      56.06        API
           100,000      12.00      56.06        API
           460,000      55.20      56.06        API
           500,000      60.00      56.06      local

  se igualan alrededor de 467,167 consultas al mes
  tu operación pro

Dos conclusiones, y la segunda es la incómoda.

La primera es sobre el cómputo: a este volumen, la API de pago sale mucho más barata que sostener
un equipo propio. El punto en que se igualan está lejísimos del tráfico real. Eso no cierra la
discusión, porque hay razones de privacidad para preferir lo local, pero sí obliga a decir la
verdad: correr el modelo en casa a esta escala es una decisión de control de los datos, no de
ahorro. Presentarla como ahorro es un error que se descubre al primer trimestre.

La segunda es que **el cómputo es una fracción pequeña del costo total**. Lo caro es el trabajo
humano: mantener los documentos al día, revisar la evaluación, atender lo que se rompa. Un
presupuesto que solo contemple el cómputo está subestimando el proyecto por mucho, y es la razón
más frecuente por la que estos sistemas se degradan a los pocos meses: nadie presupuestó a quien
tenía que cuidarlos.

Y no es un gasto opcional. Un corpus desactualizado produce respuestas incorrectas con total
seguridad, que es exactamente el fallo más caro en atención al cliente.

## 6. Qué comprar y qué construir

Un sistema en producción necesita más piezas de las que tiene tu piloto: extracción de contenido
de formatos variados, procesamiento de tablas e imágenes, reordenamiento de resultados, detección
de alucinaciones, cifrado y control de accesos, monitoreo.

Cada pieza se puede construir o contratar, y la decisión no es solo de precio. Construir todo
deja una arquitectura frágil donde tú eres responsable de cada conexión; contratar todo deja una
dependencia y una factura. Lo normal es una mezcla, y conviene tomarla pieza por pieza con
criterios explícitos.

In [6]:
# 1 = poco, 5 = mucho. "Diferenciador" es cuánto de tu ventaja depende de esa pieza.
# Cada pieza con tres notas del 1 al 5: cuánto cuesta construirla, cuánto mantenerla,
# y cuánto de la ventaja de la empresa depende de ella. La tercera es la que decide.
PIEZAS = [
    ("Extracción de PDF y Word",   2, 4, 1),
    ("Procesar tablas e imágenes", 4, 3, 1),
    ("Base vectorial",             2, 2, 1),
    ("Reordenamiento",             3, 3, 2),
    ("Curaduría del corpus",       3, 5, 5),
    ("Gold set y evaluación",      2, 5, 5),
    ("Redacción de datos locales", 3, 4, 4),
    ("Control de acceso por rol",  4, 3, 2),
    ("Monitoreo y bitácora",       3, 3, 2),
    ("Interfaz de chat",           2, 2, 3),
]

print(f"{'pieza':<28} {'construir':>10} {'mantener':>9} {'nos':>6}   {'recomendación'}")
print(f"{'':<28} {'cuesta':>10} {'cuesta':>9} {'diferencia':>6}")
print("-" * 92)
# La regla en tres renglones: si diferencia, se construye aunque cueste; si cuesta y
# no diferencia, se contrata; si es barato, se construye porque no vale la negociación.
for pieza, construir, mantener, diferencia in PIEZAS:
    esfuerzo = construir + mantener
    if diferencia >= 4:
        rec = "construir: es la ventaja"
    elif esfuerzo >= 7:
        rec = "contratar: cuesta y no diferencia"
    elif esfuerzo <= 4:
        rec = "construir: es barato"
    else:
        rec = "evaluar según el proveedor"
    print(f"{pieza:<28} {construir:>10} {mantener:>9} {diferencia:>6}   {rec}")

print("\n\nLo que conviene construir, porque es donde está la ventaja:")
for pieza, _, _, d in PIEZAS:
    if d >= 4:
        print(f"   {pieza}")

pieza                         construir  mantener    nos   recomendación
                                 cuesta    cuesta diferencia
--------------------------------------------------------------------------------------------
Extracción de PDF y Word              2         4      1   evaluar según el proveedor
Procesar tablas e imágenes            4         3      1   contratar: cuesta y no diferencia
Base vectorial                        2         2      1   construir: es barato
Reordenamiento                        3         3      2   evaluar según el proveedor
Curaduría del corpus                  3         5      5   construir: es la ventaja
Gold set y evaluación                 2         5      5   construir: es la ventaja
Redacción de datos locales            3         4      4   construir: es la ventaja
Control de acceso por rol             4         3      2   contratar: cuesta y no diferencia
Monitoreo y bitácora                  3         3      2   evaluar según el proveed

El patrón que sale de la tabla es constante en este tipo de proyectos, y vale la pena nombrarlo.

Lo que no te diferencia conviene contratarlo. Extraer texto de un PDF es un problema resuelto que
alguien más mantiene mejor que tú, y hacerlo en casa consume tiempo sin que ningún cliente lo
note.

Lo que sí te diferencia hay que construirlo, y resulta que no es la tecnología. Es la curaduría
del corpus, el gold set y las reglas de tu negocio. Nadie te va a vender un conjunto de preguntas
etiquetadas sobre tus políticas, ni va a saber que tus clientes dicen "plata" en vez de
"reembolso". Ahí es donde el trabajo propio rinde, y es justo lo que suele quedar sin
presupuesto porque no parece tecnología.

In [7]:
# Las preguntas que hay que hacerle a cualquier proveedor ANTES de firmar. Las dos
# más importantes están en Seguridad: qué hacen con los datos que reciben y si se
# pueden recuperar los propios al irse.
CHECKLIST = {
    "Interfaz y conexión": [
        "¿Ofrecen una interfaz estable y documentada?",
        "¿Cuál es el límite de llamadas por minuto? ¿Se puede procesar por lotes?",
        "¿Cómo se autentica el servicio?",
    ],
    "Datos y formatos": [
        "¿Qué formatos acepta y en cuál devuelve el resultado?",
        "¿La salida entra directo al siguiente componente o hay que transformarla?",
        "¿Se pueden exportar los datos si decidimos irnos?",
    ],
    "Seguridad": [
        "¿Cómo se cifran los datos en tránsito y almacenados?",
        "¿Qué hacen con los datos personales que reciben?",
        "¿Cumplen la normativa de protección de datos que nos aplica?",
        "¿Los datos que enviamos se usan para entrenar sus modelos?",
    ],
    "Desempeño": [
        "¿Qué tiempo de respuesta garantizan para el 95% de las llamadas?",
        "¿Cómo escalan cuando sube la carga?",
        "¿Qué disponibilidad garantizan y qué compensación hay si no la cumplen?",
    ],
    "Monitoreo": [
        "¿Hay un tablero de estado del servicio?",
        "¿Se integra con nuestras herramientas de monitoreo?",
    ],
    "Soporte": [
        "¿Por qué canales se pide soporte?",
        "¿En cuánto tiempo responden un problema que detiene la operación?",
        "¿Cómo avisan de los cambios de versión?",
    ],
}

print("PREGUNTAS PARA CUALQUIER PROVEEDOR\n")
n = 0
for area, preguntas in CHECKLIST.items():
    print(f"{area}")
    for p in preguntas:
        n += 1
        print(f"   {n:>2}. {p}")
    print()
print(f"{n} preguntas. Conviene hacerlas por escrito y guardar las respuestas:")
print("cuando algo falle, esa conversación es lo que define de quién es el problema.")

PREGUNTAS PARA CUALQUIER PROVEEDOR

Interfaz y conexión
    1. ¿Ofrecen una interfaz estable y documentada?
    2. ¿Cuál es el límite de llamadas por minuto? ¿Se puede procesar por lotes?
    3. ¿Cómo se autentica el servicio?

Datos y formatos
    4. ¿Qué formatos acepta y en cuál devuelve el resultado?
    5. ¿La salida entra directo al siguiente componente o hay que transformarla?
    6. ¿Se pueden exportar los datos si decidimos irnos?

Seguridad
    7. ¿Cómo se cifran los datos en tránsito y almacenados?
    8. ¿Qué hacen con los datos personales que reciben?
    9. ¿Cumplen la normativa de protección de datos que nos aplica?
   10. ¿Los datos que enviamos se usan para entrenar sus modelos?

Desempeño
   11. ¿Qué tiempo de respuesta garantizan para el 95% de las llamadas?
   12. ¿Cómo escalan cuando sube la carga?
   13. ¿Qué disponibilidad garantizan y qué compensación hay si no la cumplen?

Monitoreo
   14. ¿Hay un tablero de estado del servicio?
   15. ¿Se integra con nuestras 

De esa lista, dos preguntas merecen atención especial porque son las que más se olvidan y las que
más caro salen.

La de si los datos que envías se usan para entrenar sus modelos: en atención al cliente estás
mandando consultas que contienen datos personales, y la respuesta a esa pregunta determina si
puedes usar el servicio o no.

La de si puedes exportar tus datos: es la que decide si dentro de dos años tienes opción de
cambiar de proveedor o estás atrapado. Se pregunta al principio, cuando todavía tienes poder de
negociación, no cuando ya migraste todo.

Y una observación sobre el soporte que se aprende con el tiempo. Si contratas cinco proveedores,
cuando algo falle vas a tener cinco mesas de ayuda distintas y ninguna se va a hacer cargo de la
integración. Coordinar eso es un trabajo real que hay que presupuestar, y es el argumento más
fuerte a favor de reducir el número de proveedores aunque cada uno individualmente parezca mejor.

## 7. El equipo

Un sistema como este toca cuatro áreas de conocimiento distintas, y es raro que una sola persona
las cubra todas. Vale la pena escribir qué se necesita y quién lo va a hacer, aunque la respuesta
sea que una misma persona lleva varios sombreros.

In [8]:
# Quién hace falta. Fíjese en cuál lleva la dedicación más alta: no es un perfil
# técnico, es el que sabe qué pregunta el cliente y qué respuesta es correcta.
PERFILES = [
    ("Ingeniería de datos",
     "conectar las fuentes, extraer y normalizar los documentos, mantener la ingesta",
     "media"),
    ("Aprendizaje automático",
     "elegir y evaluar modelos, ajustar la recuperación, medir la calidad",
     "media"),
    ("Operaciones y despliegue",
     "poner el sistema a correr, monitorearlo, atender caídas",
     "baja"),
    ("Seguridad y cumplimiento",
     "datos personales, control de accesos, bitácora, normativa aplicable",
     "media"),
    ("Conocimiento del negocio",
     "qué preguntan los clientes, qué respuesta es correcta, qué no se puede decir",
     "alta"),
]

print(f"{'área':<28} {'qué hace':<62} {'dedicación'}")
print("-" * 104)
for area, tarea, carga in PERFILES:
    print(f"{area:<28} {tarea:<62} {carga}")

print("\nDedicación alta significa que necesita a alguien de forma sostenida;")
print("media, algunas horas por semana; baja, atención puntual.")

área                         qué hace                                                       dedicación
--------------------------------------------------------------------------------------------------------
Ingeniería de datos          conectar las fuentes, extraer y normalizar los documentos, mantener la ingesta media
Aprendizaje automático       elegir y evaluar modelos, ajustar la recuperación, medir la calidad media
Operaciones y despliegue     poner el sistema a correr, monitorearlo, atender caídas        baja
Seguridad y cumplimiento     datos personales, control de accesos, bitácora, normativa aplicable media
Conocimiento del negocio     qué preguntan los clientes, qué respuesta es correcta, qué no se puede decir alta

Dedicación alta significa que necesita a alguien de forma sostenida;
media, algunas horas por semana; baja, atención puntual.


La fila con más dedicación es la única que no es técnica.

No es un descuido de la tabla. Quien sabe qué preguntan los clientes, cuál es la respuesta
correcta y qué no se puede decir por escrito es la persona que sostiene la calidad del sistema, y
su trabajo no termina nunca: cada cambio de política, cada producto nuevo, cada campaña genera
preguntas que el corpus todavía no cubre.

En la práctica esto suele resolverse mal. Se arma un equipo técnico, se lanza el sistema, y la
parte de negocio se atiende con lo que sobra del tiempo de alguien que además tiene otro puesto.
Seis meses después el corpus está desactualizado y el sistema responde con políticas que ya no
existen.

Si de este taller te llevas una sola cosa a la hora de armar el equipo, que sea esta: la persona
que cuida el contenido no es opcional y no es media plaza prestada.

## 8. Qué puede salir mal

Un plan sin riesgos escritos no es un plan, es un deseo. Los riesgos de estos proyectos son
bastante predecibles, así que conviene anotarlos antes con su señal de alerta y su respuesta.

In [9]:
# Cada riesgo con cuatro datos: qué tan probable es, qué tan grave sería, por qué
# señal se daría cuenta, y qué se hace al respecto. Un riesgo sin señal de alerta no
# se puede vigilar, y uno sin respuesta no es un plan.
RIESGOS = [
    ("El corpus se desactualiza",
     "alta", "alto",
     "sube la proporción de respuestas incorrectas sobre temas recientes",
     "responsable asignado y revisión mensual del contenido"),
    ("El sistema inventa una política",
     "media", "muy alto",
     "un cliente reclama citando una respuesta del bot",
     "abstención probada, citas visibles y revisión semanal de la bitácora"),
    ("Se filtra un dato personal",
     "baja", "muy alto",
     "aparece un dato de cliente en una respuesta o en el registro",
     "redacción medida sobre casos reales y filtro por rol antes de buscar"),
    ("Los clientes dejan de usarlo",
     "media", "alto",
     "el volumen de consultas cae tras las primeras semanas",
     "vigilar el uso diario el primer mes y preguntar por qué a quien dejó de usarlo"),
    ("Las respuestas tardan demasiado",
     "media", "medio",
     "el tiempo del percentil 95 pasa de la meta",
     "caché de consultas frecuentes y alerta automática de tiempo"),
    ("El proveedor cambia precios o condiciones",
     "media", "medio",
     "aviso de cambio contractual",
     "no depender de un solo proveedor para la pieza crítica"),
    ("Nadie revisa la evaluación",
     "alta", "medio",
     "pasan semanas sin que se corra el gold set",
     "evaluación automática semanal con resultado enviado por correo"),
]

# Para poder ordenar hay que convertir las palabras en números.
peso = {"baja": 1, "media": 2, "alta": 3,
        "medio": 2, "alto": 3, "muy alto": 4}

# sorted ordena la lista; key le dice según qué. Aquí es probabilidad por impacto, y
# el signo menos invierte el orden para que lo más grave quede arriba.
ordenados = sorted(RIESGOS, key=lambda r: -(peso[r[1]] * peso[r[2]]))

print("RIESGOS, ordenados por probabilidad e impacto\n")
for riesgo, prob, impacto, senal, respuesta in ordenados:
    print(f"[{peso[prob]*peso[impacto]:>2}] {riesgo}  (probabilidad {prob}, impacto {impacto})")
    print(f"      señal de alerta: {senal}")
    print(f"      qué hacemos    : {respuesta}")
    print()

RIESGOS, ordenados por probabilidad e impacto

[ 9] El corpus se desactualiza  (probabilidad alta, impacto alto)
      señal de alerta: sube la proporción de respuestas incorrectas sobre temas recientes
      qué hacemos    : responsable asignado y revisión mensual del contenido

[ 8] El sistema inventa una política  (probabilidad media, impacto muy alto)
      señal de alerta: un cliente reclama citando una respuesta del bot
      qué hacemos    : abstención probada, citas visibles y revisión semanal de la bitácora

[ 6] Los clientes dejan de usarlo  (probabilidad media, impacto alto)
      señal de alerta: el volumen de consultas cae tras las primeras semanas
      qué hacemos    : vigilar el uso diario el primer mes y preguntar por qué a quien dejó de usarlo

[ 6] Nadie revisa la evaluación  (probabilidad alta, impacto medio)
      señal de alerta: pasan semanas sin que se corra el gold set
      qué hacemos    : evaluación automática semanal con resultado enviado por correo

[ 4] S

Los dos primeros de la lista comparten una característica que conviene ver: **no son fallas
técnicas**. El corpus que se desactualiza y la evaluación que nadie corre son fallas de proceso,
y ocurren en sistemas que funcionan perfectamente el día que se lanzan.

Por eso la respuesta a los dos es la misma y no es tecnológica: alguien responsable y una fecha.

De los riesgos técnicos, el que hay que vigilar primero es la caída de uso. Es la señal más
temprana de que algo no sirve, y la más fácil de pasar por alto porque no produce ningún error.
Si el volumen sube las primeras semanas y luego cae, los clientes probaron el sistema y
volvieron al canal de antes. Ahí hay que ir a preguntar por qué, y la respuesta casi nunca está
en los registros del sistema.

## 9. El plan en un documento

Esta última celda junta todo lo anterior en un archivo de texto que puedes abrir, editar y usar
como base de una propuesta real.

Está pensado para que lo adaptes: cambia los números de la sección de operación, ajusta las metas
de la tabla de requisitos con las de tu propio caso, y quita las filas que no apliquen.

In [10]:
from datetime import date
from pathlib import Path

# Lo que sigue arma el documento del plan juntando todo lo anterior. La gracia es que
# el documento se genera desde las mismas cifras que se midieron: si cambia una
# medición, se vuelve a ejecutar y el plan queda al día solo.
lineas = []
# Un atajo para no escribir lineas.append() cien veces. A("texto") agrega una línea.
A = lineas.append

A("# Plan de paso a producción")
A("")
A("Sistema de respuesta automática sobre documentación de atención al cliente.")
A(f"Documento generado el {date.today().isoformat()} a partir de las mediciones del piloto.")
A("")

A("## 1. Qué se construyó en el piloto")
A("")
A("| Componente | Elección | Por qué |")
A("|---|---|---|")
for parte, eleccion, razon in COMPONENTES:
    A(f"| {parte} | {eleccion} | {razon} |")
A("")

A("## 2. Qué se midió")
A("")
A(f"- Corpus de {PILOTO['documentos']} documentos y {PILOTO['fragmentos']} fragmentos.")
A(f"- Tiempo por consulta: {total_ms/1000:.1f} s, de los cuales "
  f"{PILOTO['ms_generar']/total_ms*100:.0f}% se va en redactar la respuesta.")
A(f"- Precisión de la búsqueda {PILOTO['precision_k3']:.0%}, "
  f"cobertura {PILOTO['cobertura_k3']:.0%}.")
A(f"- Respuestas sin nada incorrecto: {PILOTO['sin_nada_incorrecto']:.0%}.")
A(f"- Se abstiene cuando no puede responder en {PILOTO['se_abstiene']:.0%} de los casos probados.")
A("")
A("La conclusión operativa de estas cifras es que la recuperación funciona mejor que la")
A("redacción de la respuesta, así que el esfuerzo de mejora va en la segunda.")
A("")

A("## 3. Requisitos para producción")
A("")
A("| Aspecto | Hoy | Meta | Prioridad |")
A("|---|---|---|---|")
for aspecto, hoy, meta, prioridad in REQUISITOS:
    A(f"| {aspecto} | {hoy} | {meta} | {prioridad} |")
A("")

A("## 4. Dimensionamiento")
A("")
A(f"- {consultas_mes:,} consultas al mes, {pico_por_hora:.0f} en la hora pico.")
A(f"- {pico_por_segundo:.2f} consultas por segundo en el pico: no requiere arquitectura")
A("  distribuida a esta escala.")
A(f"- Índice de {fragmentos:,} fragmentos.")
A(f"- Reconstrucción completa: {reindexado:.0f} s. Actualización incremental: "
  f"{incremental:.1f} s.")
A("  Se usa la incremental para no dejar el índice a medias mientras se reconstruye.")
A("")

A("## 5. Costo mensual estimado")
A("")
A("| Concepto | USD |")
A("|---|---|")
A(f"| Cómputo (opción más barata al volumen previsto) | {menor:.2f} |")
A(f"| Trabajo humano ({horas} h/mes) | {personas_mes:.2f} |")
A(f"| **Total** | **{menor + personas_mes:.2f}** |")
A("")
A(f"El cómputo representa el {menor/(menor+personas_mes)*100:.1f}% del costo. El resto es")
A("trabajo de personas, que es la partida que suele quedar fuera del presupuesto.")
A("")

A("## 6. Construir o contratar")
A("")
A("Se construye lo que constituye la ventaja propia:")
A("")
for pieza, _, _, d in PIEZAS:
    if d >= 4:
        A(f"- {pieza}")
A("")
A("El resto se contrata, salvo que el proveedor no responda satisfactoriamente el")
A(f"cuestionario de {n} preguntas incluido en el anexo.")
A("")

A("## 7. Equipo")
A("")
A("| Área | Responsabilidad | Dedicación |")
A("|---|---|---|")
for area, tarea, carga in PERFILES:
    A(f"| {area} | {tarea} | {carga} |")
A("")

A("## 8. Riesgos")
A("")
A("| Riesgo | Probabilidad | Impacto | Señal de alerta | Respuesta |")
A("|---|---|---|---|---|")
for riesgo, prob, impacto, senal, respuesta in ordenados:
    A(f"| {riesgo} | {prob} | {impacto} | {senal} | {respuesta} |")
A("")

A("## 9. Criterios de decisión")
A("")
A("El paso a producción se autoriza cuando se cumplan los requisitos marcados como")
A("críticos. El sistema se retira si, tras dos meses en operación, la proporción de")
A("consultas resueltas sin intervención humana no supera la del canal que reemplaza.")
A("")

A("## Anexo. Cuestionario para proveedores")
A("")
for area, preguntas in CHECKLIST.items():
    A(f"**{area}**")
    A("")
    for p in preguntas:
        A(f"- {p}")
    A("")

# El archivo queda en la carpeta del cuaderno, en formato Markdown: se abre en
# cualquier editor y se pega en Word conservando las tablas.
ruta = Path("plan_produccion.md")
ruta.write_text("\n".join(lineas), encoding="utf-8")

print(f"Escrito en {ruta.resolve()}")
print(f"{len(lineas)} líneas, {len('.'.join(lineas))/1024:.1f} KB\n")
print("Primeras líneas del documento:\n")
print("\n".join(lineas[:24]))

Escrito en /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/plan_produccion.md
143 líneas, 6.8 KB

Primeras líneas del documento:

# Plan de paso a producción

Sistema de respuesta automática sobre documentación de atención al cliente.
Documento generado el 2026-08-07 a partir de las mediciones del piloto.

## 1. Qué se construyó en el piloto

| Componente | Elección | Por qué |
|---|---|---|
| Modelo de lenguaje | gemma3:4b, local con Ollama | Elegido para que corra en cualquier laptop del curso |
| Modelo de embeddings | embeddinggemma:300m | Único de los tres probados que cruza español e inglés |
| Base vectorial | LanceDB, embebida en disco | Sin servidor que administrar; suficiente a esta escala |
| Fragmentación | recursiva, 500 caracteres con 100 de traslape | La que menos supuestos hace sobre el formato del documento |
| Búsqueda | vectorial densa, k=3 | La híbrida se probó y no mejoró con este corpus |
| Guardrails | instrucción de abstención y filtro determinista | La 

El archivo queda junto al cuaderno, con el nombre `plan_produccion.md`. Se abre con cualquier
editor de texto y se puede pegar en un procesador de textos conservando los títulos y las tablas.

Fíjate en la sección 9, la de criterios de decisión. Es la más corta y la más difícil de escribir,
porque obliga a comprometerse por adelantado con qué se va a considerar un fracaso.

Vale la pena hacerlo igual. Sin ese criterio escrito, un sistema que no funciona se sostiene por
inercia durante años: siempre hay una mejora pendiente que justifica esperar un trimestre más.
Con el criterio escrito de antemano, la conversación se vuelve una comparación con algo acordado
cuando nadie estaba defendiendo su trabajo.

## 10. Lo que te llevas

**El piloto es la fuente de datos, no un ensayo.** Todo lo que mediste sirve de primera columna
en la tabla de requisitos. Un piloto sin mediciones obliga a planear con supuestos.

**Las metas se escriben en números.** "Que responda rápido" no se puede verificar ni discutir.
"El 95% en menos de 3 segundos" sí, y obliga a acordar si ese número es el correcto.

**No todas las metas son subir.** La cobertura de la búsqueda ya estaba alta y el problema estaba
en la redacción de las respuestas. Saber dónde no invertir vale tanto como saber dónde invertir,
y solo se sabe midiendo por etapas.

**La escala real suele ser menor de lo que se imagina.** Ochocientas consultas diarias son menos
de una por segundo en el pico. Diseñar para un volumen que no va a llegar es caro y frecuente.

**El cómputo es la parte barata.** Lo caro es el trabajo humano de mantener el corpus y revisar la
evaluación. Un presupuesto que solo cuenta servidores subestima el proyecto, y esa omisión es la
causa más común de que estos sistemas se degraden.

**Lo que te diferencia no es la tecnología.** Es el corpus curado, el gold set y las reglas de tu
negocio. Eso se construye; el resto conviene contratarlo.

**Los riesgos más probables no son técnicos.** El corpus que se desactualiza y la evaluación que
nadie corre ocurren en sistemas que funcionan perfectamente el día del lanzamiento. Se atienden
con un responsable y una fecha, no con código.

Con esto cierra el recorrido: empezaste cargando un documento y respondiendo una pregunta, y
terminas con un plan para sostener el sistema en operación. La parte técnica fue la primera
mitad; la que decide si el proyecto sobrevive es esta.